In [6]:
import os
import json
from pathlib import Path
import pandas as pd

In [12]:
DATA_DIR = Path.cwd() / "data"

# Read all data files present in DATA_DIR (non-recursive), sorted for stable output
FILES = sorted(
    [p.name for p in DATA_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".jsonl", ".json", ".csv"}]
)

if not FILES:
    raise FileNotFoundError(f"No supported data files found in: {DATA_DIR}")


In [15]:
def _load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {i} in {path.name}: {e}") from e
    return pd.DataFrame(rows)

def _load_json(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        return pd.DataFrame(obj)
    if isinstance(obj, dict):
        return pd.json_normalize(obj)
    raise ValueError(f"Unsupported JSON structure in {path.name}: expected list or dict")

def _load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)

def _load_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".jsonl":
        return _load_jsonl(path)
    if suffix == ".json":
        return _load_json(path)
    if suffix == ".csv":
        return _load_csv(path)
    raise ValueError(f"Unsupported file type: {path.name}")

def explore_file(filename: str) -> dict:
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    df = _load_jsonl(path)

    info = {
        "file": filename,
        "path": str(path),
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "column_names": list(df.columns),
        "summary": _summarize_df(df),
        "head": df.head(5),
    }
    return info

def _safe_unique_count(s: pd.Series) -> int:
    """
    Count unique values even if the series contains unhashable types (list/dict).
    """
    try:
        return int(s.nunique(dropna=True))
    except TypeError:
        def _to_hashable(x):
            # Must check for list/dict FIRST — pd.isna() on a list returns an array,
            # which raises ValueError when used in a boolean context.
            if isinstance(x, (list, dict)):
                # Stable string representation for nested JSON structures
                return json.dumps(x, sort_keys=True, ensure_ascii=False)
            try:
                if pd.isna(x):
                    return None
            except (TypeError, ValueError):
                # Fallback: stringify anything else that can't be NA-checked
                return str(x)
            return x

        return int(s.map(_to_hashable).nunique(dropna=True))

def _summarize_df(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isna().sum()
    non_null = df.notna().sum()

    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_values": missing,
        "missing_pct": (missing / len(df) * 100).round(2) if len(df) else 0.0,
        "non_null": non_null,
        "unique": [ _safe_unique_count(df[c]) for c in df.columns ],
    }, index=df.columns).sort_index()

    return summary

def check_columns_consistency(results: list[dict], file_suffix: str) -> None:
    group = [r for r in results if r["file"].endswith(file_suffix)]
    print(f"\nSchema check for *{file_suffix} files")

    if not group:
        print("  No matching files found.")
        return
    if len(group) == 1:
        print(f"  Only one matching file ({group[0]['file']}); nothing to compare.")
        return

    baseline_file = group[0]["file"]
    baseline_cols = list(group[0]["column_names"])
    baseline_set = set(baseline_cols)

    same_sets = True
    same_order = True

    for r in group[1:]:
        cols = list(r["column_names"])
        cols_set = set(cols)

        if cols_set != baseline_set:
            same_sets = False
            missing = sorted(baseline_set - cols_set)
            extra = sorted(cols_set - baseline_set)
            print(f"  MISMATCH: {r['file']} vs {baseline_file}")
            print(f"    Missing columns: {missing if missing else 'None'}")
            print(f"    Extra columns:   {extra if extra else 'None'}")
        elif cols != baseline_cols:
            same_order = False
            print(f"  ORDER DIFFERENCE: {r['file']} has same columns but different order.")

    if same_sets and same_order:
        print(f"  OK: all {len(group)} files have identical columns and order.")
    elif same_sets:
        print(f"  OK (set): all {len(group)} files have the same columns, but order differs.")
    else:
        print(f"  FAIL: not all {len(group)} files share the same column set.")



In [16]:
results = []
for fn in FILES:
    res = explore_file(fn)
    results.append(res)

# Pretty print results per file
for res in results:
    print("=" * 100)
    print(f"FILE: {res['file']}")
    print(f"PATH: {res['path']}")
    print(f"SHAPE: {res['rows']} rows x {res['columns']} columns")
    print(f"COLUMNS: {res['column_names']}")
    print("-" * 100)
    display(res["summary"])
    print("-" * 100)
    display(res["head"])

check_columns_consistency(results, "_data.jsonl")
check_columns_consistency(results, "_label.jsonl")

FILE: en_test_data.jsonl
PATH: C:\Users\Jekaterina.Sergejeva\OneDrive - Euromonitor International\Desktop\SHROOM-CAP-EN-LT\data\en_test_data.jsonl
SHAPE: 240 rows x 15 columns
COLUMNS: ['index', 'title', 'abstract', 'doi', 'url', 'extracted', 'datafile', 'authors', 'question', 'model_id', 'model_config', 'prompt', 'output_text', 'output_tokens', 'output_logits']
----------------------------------------------------------------------------------------------------


,dtype,missing_values,missing_pct,non_null,unique
abstract,str,0,0.0,240,30
authors,object,0,0.0,240,30
datafile,str,0,0.0,240,2
doi,str,0,0.0,240,30
extracted,bool,0,0.0,240,2
index,str,0,0.0,240,240
model_config,str,0,0.0,240,3
model_id,str,0,0.0,240,2
output_logits,object,0,0.0,240,227
output_text,str,0,0.0,240,224


----------------------------------------------------------------------------------------------------


,index,title,abstract,doi,url,extracted,datafile,authors,question,model_id,model_config,prompt,output_text,output_tokens,output_logits
0,en-tst-0,Pretraining Data Detection for Large Language ...,As the scale of training corpora for large lan...,10.18653/v1/2024.emnlp-main.300,https://github.com/zhang-wei-chao/DC-PDD,False,2024.emnlp.xml,"[{'first': 'Weichao', 'last': 'Zhang'}, {'firs...",What is the name of the Chinese-language bench...,lmsys/vicuna-7b-v1.5,k50_p0.95_t0.2,"In the article titled ""Pretraining Data Detect...",2. What is the name of the benchmark that is c...,"[<0x0A>, 2, ., ▁What, ▁is, ▁the, ▁name, ▁of, ▁...","[18.2342338562, 12.4345569611, 19.5700969696, ..."
1,en-tst-1,Formality is Favored: Unraveling the Learning ...,Having been trained on massive pretraining dat...,10.18653/v1/2024.emnlp-main.304,https://aclanthology.org/2024.emnlp-main.304.pdf,True,2024.emnlp.xml,"[{'first': 'Jiahuan', 'last': 'Li'}, {'first':...",What five characteristics are associated as in...,lmsys/vicuna-7b-v1.5,k50_p0.95_t0.2,"In the article titled ""Formality is Favored: U...",(1) CORA (2) Citeseer (3) Pubmed (4) Web of Sc...,"[<0x0A>, ▁(, 1, ), ▁C, OR, A, ▁(, 2, ), ▁Cit, ...","[18.3630599976, 13.0511350632, 13.3475074768, ..."
2,en-tst-5,"An image speaks a thousand words, but can ever...","Given the rise of multimedia content, human tr...",10.18653/v1/2024.emnlp-main.573,https://aclanthology.org/2024.emnlp-main.573.pdf,True,2024.emnlp.xml,"[{'first': 'Simran', 'last': 'Khanuja'}, {'fir...",Name the three pipelines for image transcreation.,lmsys/vicuna-7b-v1.5,k50_p0.95_t0.2,"In the article titled ""An image speaks a thous...",The article discusses the challenges of image ...,"[<0x0A>, <0x0A>, The, ▁article, ▁discuss, es, ...","[21.6078338623, 20.6981983185, 15.7419433594, ..."
3,en-tst-6,Fishing for Magikarp: Automatically Detecting ...,The disconnect between tokenizer creation and ...,10.18653/v1/2024.emnlp-main.649,https://aclanthology.org/2024.emnlp-main.649.pdf,True,2024.emnlp.xml,"[{'first': 'Sander', 'last': 'Land'}, {'first'...",What are UNREACHABLE TOKENS?,lmsys/vicuna-7b-v1.5,k50_p0.95_t0.2,"In the article titled ""Fishing for Magikarp: A...",(https://arxiv.org/abs/2105.14599),"[<0x0A>, ▁(, https, ://, ar, xiv, ., org, /, a...","[21.6736106873, 12.5693969727, 12.4558324814, ..."
4,en-tst-7,SUPER: Evaluating Agents on Setting Up and Exe...,Given that Large Language Models (LLMs) have m...,10.18653/v1/2024.emnlp-main.702,https://aclanthology.org/2024.emnlp-main.702.pdf,True,2024.emnlp.xml,"[{'first': 'Ben', 'last': 'Bogin'}, {'first': ...",What is the full form of SUPER benchmark?,lmsys/vicuna-7b-v1.5,k50_p0.95_t0.2,"In the article titled ""SUPER: Evaluating Agent...",I couldn't find it in the article.,"[<0x0A>, I, ▁couldn, ', t, ▁find, ▁it, ▁in, ▁t...","[23.3552093506, 12.4487380981, 15.2110424042, ..."


FILE: en_train2_data.jsonl
PATH: C:\Users\Jekaterina.Sergejeva\OneDrive - Euromonitor International\Desktop\SHROOM-CAP-EN-LT\data\en_train2_data.jsonl
SHAPE: 84 rows x 15 columns
COLUMNS: ['index', 'title', 'abstract', 'doi', 'url', 'extracted', 'datafile', 'authors', 'question', 'model_id', 'model_config', 'prompt', 'output_text', 'output_tokens', 'output_logits']
----------------------------------------------------------------------------------------------------


...

FILE: en_valid_label.jsonl
PATH: C:\Users\Jekaterina.Sergejeva\OneDrive - Euromonitor International\Desktop\SHROOM-CAP-EN-LT\data\en_valid_label.jsonl
SHAPE: 240 rows x 3 columns
COLUMNS: ['index', 'has_fluency_mistakes', 'has_factual_mistakes']
----------------------------------------------------------------------------------------------------


,dtype,missing_values,missing_pct,non_null,unique
has_factual_mistakes,str,0,0.0,240,2
has_fluency_mistakes,str,0,0.0,240,2
index,str,0,0.0,240,240


----------------------------------------------------------------------------------------------------


,index,has_fluency_mistakes,has_factual_mistakes
0,en-val-0,n,n
1,en-val-3,n,y
2,en-val-4,n,y
3,en-val-5,n,y
4,en-val-6,n,y



Schema check for *_data.jsonl files
  OK: all 4 files have identical columns and order.

Schema check for *_label.jsonl files
  OK: all 3 files have identical columns and order.


In [18]:
check_columns_consistency(def build_group_schema(results: list[dict], file_suffix: str) -> dict[str, str]:
    group = [r for r in results if r["file"].endswith(file_suffix)]
    if not group:
        return {}

    # Build schema from first matching file: {column_name: dtype}
    first = group[0]
    first_summary = first["summary"]
    schema = {
        col: str(first_summary.loc[col, "dtype"])
        for col in first["column_names"]
    }

    # Optional strictness: ensure all files in the group have the same column->dtype mapping
    for r in group[1:]:
        s = r["summary"]
        current = {col: str(s.loc[col, "dtype"]) for col in r["column_names"]}
        if current != schema:
            raise ValueError(
                f"Schema mismatch in {r['file']} for suffix {file_suffix}. "
                "Column names/types are not identical across files."
            )

    return schema


def save_group_schema(results: list[dict], file_suffix: str, output_filename: str) -> Path:
    schema = build_group_schema(results, file_suffix)
    output_path = DATA_DIR / output_filename

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(schema, f, indent=2, ensure_ascii=False)

    print(f"Schema saved: {output_path}")
    return output_pathresults, "_data.jsonl")
check_columns_consistency(results, "_label.jsonl")

save_group_schema(results, "_data.jsonl", "schema_data.json")
save_group_schema(results, "_label.jsonl", "schema_label.json")


Schema check for *_data.jsonl files
  OK: all 4 files have identical columns and order.

Schema check for *_label.jsonl files
  OK: all 3 files have identical columns and order.
Schema saved: C:\Users\Jekaterina.Sergejeva\OneDrive - Euromonitor International\Desktop\SHROOM-CAP-EN-LT\data\schema_data.json
Schema saved: C:\Users\Jekaterina.Sergejeva\OneDrive - Euromonitor International\Desktop\SHROOM-CAP-EN-LT\data\schema_label.json


WindowsPath('C:/Users/Jekaterina.Sergejeva/OneDrive - Euromonitor International/Desktop/SHROOM-CAP-EN-LT/data/schema_label.json')

In [19]:
def build_group_schema(results: list[dict], file_suffix: str) -> dict[str, str]:
    group = [r for r in results if r["file"].endswith(file_suffix)]
    if not group:
        return {}

    # Build schema from first matching file: {column_name: dtype}
    first = group[0]
    first_summary = first["summary"]
    schema = {
        col: str(first_summary.loc[col, "dtype"])
        for col in first["column_names"]
    }

    # Optional strictness: ensure all files in the group have the same column->dtype mapping
    for r in group[1:]:
        s = r["summary"]
        current = {col: str(s.loc[col, "dtype"]) for col in r["column_names"]}
        if current != schema:
            raise ValueError(
                f"Schema mismatch in {r['file']} for suffix {file_suffix}. "
                "Column names/types are not identical across files."
            )

    return schema


def save_group_schema(results: list[dict], file_suffix: str, output_filename: str) -> Path:
    schema = build_group_schema(results, file_suffix)
    output_path = DATA_DIR / output_filename

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(schema, f, indent=2, ensure_ascii=False)

    print(f"Schema saved: {output_path}")
    return output_path

In [20]:
save_group_schema(results, "_data.jsonl", "schema_data.json")
save_group_schema(results, "_label.jsonl", "schema_label.json")

Schema saved: C:\Users\Jekaterina.Sergejeva\OneDrive - Euromonitor International\Desktop\SHROOM-CAP-EN-LT\data\schema_data.json
Schema saved: C:\Users\Jekaterina.Sergejeva\OneDrive - Euromonitor International\Desktop\SHROOM-CAP-EN-LT\data\schema_label.json


WindowsPath('C:/Users/Jekaterina.Sergejeva/OneDrive - Euromonitor International/Desktop/SHROOM-CAP-EN-LT/data/schema_label.json')